In [17]:
import numpy as np 
import pandas as pd
import os
from datasets import load_dataset, Dataset
from sklearn.model_selection import GroupShuffleSplit 
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
import re, torch 

#fixed seeds for reproducibility
random_state = 42

device = "cuda"

In [18]:
# load dataset from hugging face
data = load_dataset("b-mc2/sql-create-context")
 
#keep the data as a Hugging Face Dataset
ds = data["train"]

In [19]:
#splitting train/test/val without data leakage
#randomly divide data into subsets while keeping entire schema groups together
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state = random_state) #split to training and holdout(test and val)
train_idx, hold_idx = next(gss.split(np.arange(len(ds)), groups=list(ds["context"]))) #the context column defines the groups
train_raw, hold_raw = ds.select(train_idx), ds.select(hold_idx) #select rows by index (still Hugging Face datasets)
 
#split the holdout 20% of groups in 2 : val and test
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state = random_state)
val_idx, test_idx = next(gss2.split(np.arange(len(hold_raw)), groups=list(hold_raw["context"])))
val_raw, test_raw = hold_raw.select(val_idx), hold_raw.select(test_idx)
 
# [safety check, already verified] schema overlap between splits
# print(len(train_raw), len(val_raw), len(test_raw))
#
# print("Train/Validation schema overlap:",
#       len(set(train_raw["context"]) & set(val_raw["context"])))
#
# print("Train/Test schema overlap:",
#       len(set(train_raw["context"]) & set(test_raw["context"])))
#
# print("Validation/Test schema overlap:",
#       len(set(val_raw["context"]) & set(test_raw["context"])))

In [20]:
#tokenization
tok = AutoTokenizer.from_pretrained("t5-small")
 
# [diagnostic only] eyeball tokenization on one example
# def make_input(q, c):
#     return f"question: {q} context: {c}"
#
# #test tokenization on the first row of training data
# ex = train_raw[0]
# text = make_input(ex["question"], ex["context"])
# ids = tok(text).input_ids #tokenized IDs
# print(text)
# print(ids)
# print(tok.convert_ids_to_tokens(ids))

In [21]:
# [diagnostic only] check which characters the tokenizer can't represent
# #make sure the tokenizer can represent SQL characters and symbols
# sample = train_raw.shuffle(seed=0).select(range(5000))
# unk = tok.unk_token_id
#
# sample_answers = list(sample["answer"])
# all_chars = set("".join(sample_answers))
# bad_chars = [ch for ch in all_chars if ch.strip() and unk in tok(ch).input_ids]
# print("Characters the tokenizer can't represent:", bad_chars)
#
# has_unk = [unk in tok(s).input_ids for s in sample_answers]
# print(f"SQL answers affected: {np.mean(has_unk):.1%}")
# print([a for a, h in zip(sample_answers, has_unk) if h][:3])

In [22]:
# [diagnostic only] token length stats used to pick MAX_IN / MAX_OUT
# #measure the tokens in the model's input/output to choose max sequence length
# lens_in  = [len(tok(make_input(q, c)).input_ids) for q, c in zip(sample["question"], sample["context"])]
# lens_out = [len(tok(a).input_ids) for a in sample_answers]
# for name, l in [("input", lens_in), ("output", lens_out)]:
#     print(name, "median / 95% / 99% / max:", np.percentile(l, [50, 95, 99, 100]).astype(int))

In [23]:
#func to tokenize a batch of text and detect which ones contain unknown tokens
unk = tok.unk_token_id
 
def has_unk(texts):
    ids = tok(list(texts), add_special_tokens=False)["input_ids"]
    return np.array([unk in x for x in ids])

In [24]:
#replace important characters that produce unknown tokens with placeholders
PH = {
    "<": "LESSTHAN",
    "~": "TILDESYM",
    "{": "LBRACESYM",
    "}": "RBRACESYM",
    "\\": "BACKSLASHSYM",
    "`": "BACKTICKSYM",
}
 
def encode_text(s):
    for ch, p in PH.items():
        s = s.replace(ch, p)
    return s
 
def decode_text(s):
    for ch, p in PH.items():
        s = s.replace(p, ch)
    return s
 
# [safety checks, already verified]
# # 1) No placeholder may already exist in the data
# for name, d in [("train", train_raw), ("val", val_raw), ("test", test_raw)]:
#     for col in ["question", "context", "answer"]:
#         for p in PH.values():
#             assert not any(p in x for x in d[col]), f"{p} in {name}.{col}"
#
# # 2) Round trip must be exact
# for d in (train_raw, val_raw, test_raw):
#     assert all(decode_text(encode_text(a)) == a for a in d["answer"])
# print("Placeholders safe, round trip OK")
#
# # 3) Any ASCII gaps left?
# chars = set("".join(encode_text(a) for a in train_raw["answer"]))
# print("Unrepresentable ASCII:", [c for c in chars if c.isascii() and c.strip()
#       and unk in tok(c, add_special_tokens=False).input_ids])

In [25]:
# prepare train, test, val datasets
def build(d):
    d = d.map(lambda ex: {
        "input_text":  encode_text(f"question: {ex['question']} context: {ex['context']}"),
        "target_text": encode_text(ex["answer"]),
    })
    flags = has_unk(d["input_text"]) | has_unk(d["target_text"])
    return d.add_column("has_unk", flags.tolist())
 
train_b, val_b, test_b = build(train_raw), build(val_raw), build(test_raw)
train_b = train_b.filter(lambda ex: not ex["has_unk"])   # train only!
print(len(train_b), len(val_b), len(test_b))
# print("clean fraction of test:", 1 - np.mean(list(test_b["has_unk"])))

61797 7859 7823


In [26]:
#tokenize datasets and apply max sequence length
MAX_IN, MAX_OUT = 192, 192
 
def tokenize(batch):
    enc = tok(batch["input_text"], max_length=MAX_IN, truncation=True)
    enc["labels"] = tok(text_target=batch["target_text"],
                        max_length=MAX_OUT, truncation=True)["input_ids"]
    return enc
 
def to_ds(d):
    ds_ = d.select_columns(["input_text", "target_text"])
    return ds_.map(tokenize, batched=True, remove_columns=["input_text", "target_text"])
 
train_ds, val_ds, test_ds = to_ds(train_b), to_ds(val_b), to_ds(test_b)
 
# [diagnostic only] truncation counts and a decoded label sanity check
# print("Truncated inputs :", sum(len(x) == MAX_IN  for x in train_ds["input_ids"]))
# print("Truncated targets:", sum(len(x) == MAX_OUT for x in train_ds["labels"]))
# print(tok.decode(train_ds[0]["labels"], skip_special_tokens=True))

In [27]:
model0 = AutoModelForSeq2SeqLM.from_pretrained("t5-small").to(device)

def generate(model, texts, bs=64, max_new_tokens=192, num_beams=1):
    model.eval(); outs = []
    for i in range(0, len(texts), bs):
        batch = tok(texts[i:i+bs], return_tensors="pt", padding=True,
                    truncation=True, max_length=MAX_IN).to(device)
        with torch.no_grad():
            g = model.generate(**batch, max_new_tokens=max_new_tokens, num_beams=num_beams)
        outs += tok.batch_decode(g, skip_special_tokens=True,
                                 clean_up_tokenization_spaces=False)
    return outs

def norm(s):
    return re.sub(r"\s+", " ", s).strip()

sub = test_b.shuffle(seed=0).select(range(200))
preds = [decode_text(p) for p in generate(model0, sub["input_text"])]
em = np.mean([norm(p) == norm(a) for p, a in zip(preds, sub["answer"])])
print(f"Baseline exact match: {em:.1%}")
for p, a in list(zip(preds, sub["answer"]))[:5]:
    print("PRED:", p)
    print("GOLD:", a)
    print("---")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Baseline exact match: 0.0%
PRED: CREATE TABLE Movie
GOLD: SELECT T2.title, T1.rID, T1.stars, MIN(T1.stars) FROM Rating AS T1 JOIN Movie AS T2 ON T1.mID = T2.mID GROUP BY T1.rID
---
PRED: 3
GOLD: SELECT AVG(losses) FROM table_name_24 WHERE wins = 3
---
PRED: CREATE TABLE table_name_11
GOLD: SELECT venue FROM table_name_11 WHERE result = "7-2"
---
PRED: VARCHAR
GOLD: SELECT country FROM table_name_4 WHERE score < 70 AND place = "t3" AND player = "andrés romero"
---
PRED: VARCHAR
GOLD: SELECT dance FROM table_name_16 WHERE result = "safe" AND week = 1
---


In [28]:
m = AutoModelForSeq2SeqLM.from_pretrained("t5-small")
smoke = Seq2SeqTrainer(
    model=m,
    args=Seq2SeqTrainingArguments(
        output_dir="/kaggle/working/smoke", per_device_train_batch_size=32,
        learning_rate=3e-4, max_steps=100, logging_steps=20,
        save_strategy="no", eval_strategy="no", report_to="none", fp16=False),
    train_dataset=train_ds,
    data_collator=DataCollatorForSeq2Seq(tok, model=m),
    processing_class=tok,
)
smoke.train()

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
20,3.933181
40,1.458361
60,1.141002
80,0.973197
100,0.914087


TrainOutput(global_step=100, training_loss=1.683965892791748, metrics={'train_runtime': 46.9315, 'train_samples_per_second': 136.369, 'train_steps_per_second': 2.131, 'total_flos': 158552920424448.0, 'train_loss': 1.683965892791748, 'epoch': 0.10351966873706005})

In [29]:
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")

args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/t5-sql",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=3e-4,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=100,
    report_to="none",
    fp16=False,
    seed=42,
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=DataCollatorForSeq2Seq(tok, model=model),
    processing_class=tok,
)
trainer.train()

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,0.328896,0.205443
2,0.253157,0.171833
3,0.212590,0.161755


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=2898, training_loss=0.4130755477810498, metrics={'train_runtime': 1424.4979, 'train_samples_per_second': 130.145, 'train_steps_per_second': 2.034, 'total_flos': 4587210255925248.0, 'train_loss': 0.4130755477810498, 'epoch': 3.0})

In [30]:
sub = val_b.shuffle(seed=0).select(range(200))

preds = [decode_text(p) for p in generate(model, sub["input_text"])]

em = np.mean([
    norm(p) == norm(a)
    for p, a in zip(preds, sub["answer"])
])

print(f"Fine-tuned exact match (val sample): {em:.1%}")

for p, a in list(zip(preds, sub["answer"]))[:8]:
    print("PRED:", p)
    print("GOLD:", a)
    print("---")

Fine-tuned exact match (val sample): 69.0%
PRED: SELECT score_in_the_final FROM table_name_41 WHERE date = "1981"
GOLD: SELECT score_in_the_final FROM table_name_41 WHERE date = 1981
---
PRED: SELECT COUNT(rank) FROM table_name_13 WHERE celebrity = "marcin mroczek" AND season > 4
GOLD: SELECT COUNT(rank) FROM table_name_13 WHERE celebrity = "marcin mroczek" AND season > 4
---
PRED: SELECT school FROM table_11677691_3 WHERE position = "Rightback" AND college = "Nebraska"
GOLD: SELECT school FROM table_11677691_3 WHERE position = "Running back" AND college = "Nebraska"
---
PRED: SELECT grid FROM table_name_57 WHERE time_retired = "hydraulics"
GOLD: SELECT grid FROM table_name_57 WHERE time_retired = "hydraulics"
---
PRED: SELECT MAX(week) FROM table_name_69 WHERE result = "t 35-35"
GOLD: SELECT MAX(week) FROM table_name_69 WHERE result = "t 35-35"
---
PRED: SELECT digital_analog_signal FROM table_name_71 WHERE available_interface = "pci express" AND retail_name = "all-in-wonder x600 pro"

In [31]:
# full val run, bs=64 and max_new_tokens=192 to match MAX_OUT
val_preds = [decode_text(p) for p in generate(model, val_b["input_text"], bs=64, max_new_tokens=192)]
hits = np.array([norm(p) == norm(g) for p, g in zip(val_preds, val_b["answer"])])
clean = ~np.array(val_b["has_unk"])   # examples without unrepresentable characters
print(f"EM all   ({len(hits)}): {hits.mean():.1%}")
print(f"EM clean ({clean.sum()}): {hits[clean].mean():.1%}")

EM all   (7859): 67.8%
EM clean (7702): 69.2%


In [32]:
errs = pd.DataFrame({"question": val_b["question"], "context": val_b["context"],
                     "gold": val_b["answer"], "pred": val_preds})[~hits]
errs.to_csv("/kaggle/working/val_errors.csv", index=False)   # keep for README analysis
for _, r in errs.sample(15, random_state=0).iterrows():
    print("Q   :", r.question); print("PRED:", r.pred); print("GOLD:", r.gold); print("---")

Q   : What is the Losing bonus with an extra try bonus?
PRED: SELECT losing_bonus FROM table_name_89 WHERE try_bonus = "extra"
GOLD: SELECT losing_bonus FROM table_name_89 WHERE try_bonus = "try bonus"
---
Q   : Which owner has a description of Mark 1 CK and is dated 1953?
PRED: SELECT owner_s_ FROM table_name_30 WHERE date = "1953" AND description = "mark 1 kw"
GOLD: SELECT owner_s_ FROM table_name_30 WHERE date = 1953 AND description = "mark 1 ck"
---
Q   : Which Years in Orlando has a Nationality of united states, a Position of center, and a Player of jawann oldham?
PRED: SELECT years_in_orlando FROM table_name_10 WHERE nationality = "united states" AND position = "center" AND player = " jawann oldham"
GOLD: SELECT years_in_orlando FROM table_name_10 WHERE nationality = "united states" AND position = "center" AND player = "jawann oldham"
---
Q   : Is there a tapered grind on the Surestart engine?
PRED: SELECT tapered_grind FROM table_16731248_1 WHERE engine = "Sugarstart"
GOLD: SELE

In [33]:
# evaluate(): full val EM for any model / decoding setting
def evaluate(m, num_beams=1):
    preds = [decode_text(p) for p in generate(m, val_b["input_text"], num_beams=num_beams)]
    h = np.array([norm(p) == norm(g) for p, g in zip(preds, val_b["answer"])])
    print(f"beams={num_beams}: EM all {h.mean():.1%} | clean {h[clean].mean():.1%}")
    return preds, h

for nb in (1, 4):
    evaluate(model, nb)

beams=1: EM all 67.8% | clean 69.2%
beams=4: EM all 68.9% | clean 70.3%
